In [ ]:
#Shows how to read data and create + plot power spectra

In [ ]:
from tlon_util import tlon
from my_fft_tools import my_fft_2d
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
infile = '/home/nfeatherstone/surveys/modulated/work/HFJ_54_Pm_8/new_tlon_extra/q803_r0_50000000_60000000'
omega_sun = 2.87e-6
omega0=3.0*omega_sun   # rotation rate of this model

pi = np.pi
tau_omega = 2*pi/omega0  # rotation period in seconds

In [ ]:
bphi= tlon(infile)
tau = (bphi.time-bphi.time[0])/tau_omega

In [ ]:
# We can see what latitudes are contained within the file
for i in range(bphi.nlat):
    print(i, 'lat = ', bphi.lat[i])

In [ ]:
# let's choose index 7 (10 degrees for this data set)
lat_ind = 7


In [ ]:
# First, create the 2-D complex Fourier Transform, dimensioned [omega,m]
bphi_fft, omega, m = my_fft_2d(bphi.vals[lat_ind,:,:],bphi.time,bphi.phi)

In [ ]:
# Now, the 2-D power spectrum
power = bphi_fft.real**2 + bphi_fft.imag**2

In [ ]:
# It can be useful to create 1-D m and omega spectra by summing along the two axes
m_power = np.sum(power,axis=0)
omega_power = np.sum(power,axis=1)

In [ ]:
# IMPORTANT NOTE
# We define a prograde wave as possessing a positive m-value and a positive omega value
# I.e. it looks like  e^{+/-i(m phi - omega t)}
# 
# Because of this, my_fft_2d uses a different convention for positive/negative frequencies (omega) from numpy's FFT
# The result is that:
# m = 0 sits at index nm//2 (as is standard)
# omega = 0 sitts at index nomega//2-1 (not standard)
nomega = len(omega)
nm = len(m)
print(omega[nomega//2-1])
print(m[nm//2])
# Note that m should be interpreted as an integer since we gave the routine phi, instead of some physical distance

In [ ]:
# Plot the m and omega spectra
fig, ax = plt.subplots(figsize=(15,5), ncols = 2)
ax[0].plot(omega/omega0, omega_power)
ax[0].set_yscale('log')
ax[0].set_xlabel(r'$\omega/\Omega_0$')
ax[0].set_ylabel('Power')
ax[0].set_title('Power integrated over m')

ax[1].plot(m, m_power)
ax[1].set_yscale('log')
ax[1].set_xlabel('azimuthal order m')
ax[1].set_ylabel('Power')
ax[1].set_title(r'Power integrated over $\omega$')
#Due to Rayleigh's dealiasing, there is zero power in
# The first and last nm//6 points.  
# Setting the y-limits takes a little finesse if we don't
# want to see a huge range on the y-axis
m_truncate = nm-nm//2//3
ymin = m_power[m_truncate-16]
ymax = np.max(m_power*1.5)
print(ymin,ymax,m_truncate,nm)
ax[1].set_ylim([ymin,ymax])
ax[1].set_xlim([m[nm//6],m[nm-nm//6]])

In [ ]:
# Next, let's view the 2-D power spectrum

#Note that sometime it helps when visualizing to remove the power in the m= 0 and omega = 0
# You can do this via
# power[nomega//2-1,:] = 0
# power[:,nm//2] = 0
# (though you may want to copy the power array before setting columns and rows to zero)

rms = np.sqrt(np.mean(power*power))
rf = 0.5  # Set this to smaller/larger values to saturate/desaturate the image
omega_nrm = omega/omega0

fig, ax = plt.subplots(figsize=(10,10))
ax.imshow(power,aspect='auto',origin='lower', 
          extent=[m[0],m[nm-1],omega_nrm[0],omega_nrm[nomega-1]], 
          vmin = 0, vmax = rf*rms,interpolation='none')

ax.set_xlim([0,m[nm-nm//6]])
ax.set_ylabel(r'\omega/Omega_0')
ax.set_xlabel('Azimuthal Order m')
ax.set_title('Power')

plt.show()
